### Billing Document Mock

Generates synthetic **Billing Document** (\~3,500 rows) and **Billing Document Item** (\~9,000 rows) DataFrames
from a SalesOrders source.

| Target table | Rows |
|---|---|
| `h2b_dbx_billingdocument.billingdocument.billingdocument` | ~3,500 |
| `h2b_dbx_billingdocument.billingdocument.billingdocumentitem` | ~9,000 |

**Seed:** 42 

In [0]:
import random
from datetime import timedelta
from collections import defaultdict

import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DoubleType, BooleanType,
)


def generate_billing_tables(sales_order_df, sales_order_item_df):
    """
    Generate BillingDocument (~3 500 rows) and BillingDocumentItem (~9 000 rows)
    from SalesOrder source data.

    Returns
    -------
    dict  {'BillingDocument': DataFrame, 'BillingDocumentItem': DataFrame}
    """
    rng = random.Random(42)

    # ── 1. Filter qualifying orders ──────────────────────────────────────
    qualified_orders = sales_order_df.filter(
        (F.col("OverallSDProcessStatus").isin("B", "C"))
        & (F.col("OverallSDDocumentRejectionSts") != "C")
    )

    # ── 2. Aggregate item-level tax per order + grab PayerParty ──────────
    item_aggs = sales_order_item_df.groupBy("SalesOrder").agg(
        F.sum("TaxAmount").alias("_TotalTaxAmount"),
        F.first("PayerParty").alias("_PayerParty"),
    )
    orders_joined = qualified_orders.join(item_aggs, on="SalesOrder", how="left")

    order_rows = orders_joined.select(
        "SalesOrder", "CreationDate", "TotalNetAmount", "_TotalTaxAmount",
        "SalesOrganization", "TransactionCurrency", "CustomerPaymentTerms",
        "_PayerParty", "SoldToParty", "IncotermsClassification", "PaymentMethod",
    ).collect()

    # ── 3. Collect source items + index by order ───────────────────────
    qualifying_keys = list({r["SalesOrder"] for r in order_rows})
    item_rows = sales_order_item_df.filter(
        F.col("SalesOrder").isin(qualifying_keys)
    ).select(
        "SalesOrder", "Material", "Product", "Plant", "MaterialGroup",
        "ProductHierarchyNode", "NetAmount", "TaxAmount",
        "SalesOrderItemCategory", "SalesOrganization",
    ).collect()

    items_by_order = defaultdict(list)
    for it in item_rows:
        items_by_order[it["SalesOrder"]].append(it)
    all_items_pool = item_rows  # fallback pool for cloning

    # ── 4. Target volumes & cancellation set ──────────────────────────
    TARGET_BD_BASE = 3182  # ~3 500 after 10 % cancellation pairs
    num_orders = len(order_rows)
    reps_needed = -(-TARGET_BD_BASE // num_orders)  # ceil division

    cancel_count = int(TARGET_BD_BASE * 0.10)
    cancel_set = set(rng.sample(range(TARGET_BD_BASE), cancel_count))

    # ── 5. Build BillingDocument rows (cycle through source orders) ─────
    billing_docs = []  # (doc_dict, SalesOrder_key, is_S1)
    bd_counter = 1
    doc_idx = 0

    for _rep in range(reps_needed):
        for row in order_rows:
            if doc_idx >= TARGET_BD_BASE:
                break

            bd_num = f"BD-{bd_counter:06d}"
            bd_counter += 1

            billing_date = row["CreationDate"] + timedelta(days=rng.randint(3, 10))
            is_cancelled = doc_idx in cancel_set

            base = dict(
                BillingDocument=bd_num,
                BillingDocumentType="F2",
                BillingDocumentCategory="M",
                CreationDate=billing_date,
                BillingDocumentDate=billing_date,
                SalesOrganization=row["SalesOrganization"],
                TotalNetAmount=0.0,   # recalculated from items below
                TotalTaxAmount=0.0,
                TransactionCurrency=row["TransactionCurrency"],
                CustomerPaymentTerms=row["CustomerPaymentTerms"],
                PayerParty=row["_PayerParty"],
                SoldToParty=row["SoldToParty"],
                CompanyCode="CC01",
                IncotermsClassification=row["IncotermsClassification"],
                PaymentMethod=row["PaymentMethod"],
                BillingDocumentIsCancelled=is_cancelled,
                CancelledBillingDocument=None,
                AccountingPostingStatus="A" if is_cancelled else "C",
                FiscalYear=str(billing_date.year),
                FiscalPeriod=str(billing_date.month).zfill(2),
            )
            billing_docs.append((base, row["SalesOrder"], False))

            if is_cancelled:
                cancel_bd = f"BD-{bd_counter:06d}"
                bd_counter += 1
                cancel_doc = base.copy()
                cancel_doc.update(
                    BillingDocument=cancel_bd,
                    BillingDocumentType="S1",
                    BillingDocumentIsCancelled=False,
                    CancelledBillingDocument=bd_num,
                    AccountingPostingStatus="C",
                )
                billing_docs.append((cancel_doc, row["SalesOrder"], True))

            doc_idx += 1
        if doc_idx >= TARGET_BD_BASE:
            break

    # ── 6. Build BillingDocumentItem rows (~2.57 items / doc → ~9 000) ──
    billing_items = []

    for doc_dict, so_key, is_s1 in billing_docs:
        bd_num = doc_dict["BillingDocument"]
        so_items = items_by_order.get(so_key, [])

        # Target 2 or 3 items per doc (57 % get 3 → avg ≈2.57)
        target_count = 3 if rng.random() < 0.57 else 2

        selected = list(so_items[:target_count])
        while len(selected) < target_count:
            clone = rng.choice(so_items) if so_items else rng.choice(all_items_pool)
            selected.append(clone)

        doc_net_total = 0.0
        doc_tax_total = 0.0

        for idx, it in enumerate(selected):
            item_num = f"{(idx + 1) * 10:06d}"
            net = float(it["NetAmount"] or 0)
            tax = float(it["TaxAmount"] or 0)
            if is_s1:
                net, tax = -net, -tax

            doc_net_total += net
            doc_tax_total += tax

            billing_items.append(dict(
                BillingDocument=bd_num,
                BillingDocumentItem=item_num,
                Material=it["Material"],
                Product=it["Product"],
                Plant=it["Plant"],
                MaterialGroup=it["MaterialGroup"],
                ProductHierarchyNode=it["ProductHierarchyNode"],
                NetAmount=net,
                GrossAmount=net + tax,
                TaxAmount=tax,
                EligibleAmountForCashDiscount=net,
                SalesDocumentItemCategory=it["SalesOrderItemCategory"],
                SalesOrganization=it["SalesOrganization"],
            ))

        # Reconcile doc-level totals with actual item sums
        doc_dict["TotalNetAmount"] = round(doc_net_total, 2)
        doc_dict["TotalTaxAmount"] = round(doc_tax_total, 2)

    # ── 7. Create Spark DataFrames ─────────────────────────────────────
    doc_dicts = [d[0] for d in billing_docs]

    bd_schema = StructType([
        StructField("BillingDocument", StringType()),
        StructField("BillingDocumentType", StringType()),
        StructField("BillingDocumentCategory", StringType()),
        StructField("CreationDate", DateType()),
        StructField("BillingDocumentDate", DateType()),
        StructField("SalesOrganization", StringType()),
        StructField("TotalNetAmount", DoubleType()),
        StructField("TotalTaxAmount", DoubleType()),
        StructField("TransactionCurrency", StringType()),
        StructField("CustomerPaymentTerms", StringType()),
        StructField("PayerParty", StringType()),
        StructField("SoldToParty", StringType()),
        StructField("CompanyCode", StringType()),
        StructField("IncotermsClassification", StringType()),
        StructField("PaymentMethod", StringType()),
        StructField("BillingDocumentIsCancelled", BooleanType()),
        StructField("CancelledBillingDocument", StringType()),
        StructField("AccountingPostingStatus", StringType()),
        StructField("FiscalYear", StringType()),
        StructField("FiscalPeriod", StringType()),
    ])

    bdi_schema = StructType([
        StructField("BillingDocument", StringType()),
        StructField("BillingDocumentItem", StringType()),
        StructField("Material", StringType()),
        StructField("Product", StringType()),
        StructField("Plant", StringType()),
        StructField("MaterialGroup", StringType()),
        StructField("ProductHierarchyNode", StringType()),
        StructField("NetAmount", DoubleType()),
        StructField("GrossAmount", DoubleType()),
        StructField("TaxAmount", DoubleType()),
        StructField("EligibleAmountForCashDiscount", DoubleType()),
        StructField("SalesDocumentItemCategory", StringType()),
        StructField("SalesOrganization", StringType()),
    ])

    billing_doc_df = spark.createDataFrame(doc_dicts, schema=bd_schema)
    billing_item_df = spark.createDataFrame(billing_items, schema=bdi_schema)

    return {"BillingDocument": billing_doc_df, "BillingDocumentItem": billing_item_df}

In [0]:
# Read source tables
sales_order_df = spark.table("h2b_dbx_salesorder.salesorder.salesorder")
sales_order_item_df = spark.table("h2b_dbx_salesorder.salesorder.salesorderitem")

# Generate billing tables
result = generate_billing_tables(sales_order_df, sales_order_item_df)

billing_doc_df = result["BillingDocument"]
billing_item_df = result["BillingDocumentItem"]

print(f"BillingDocument rows:     {billing_doc_df.count()}")
print(f"BillingDocumentItem rows: {billing_item_df.count()}")
print()
print("── BillingDocument sample ──")
display(billing_doc_df)
print("── BillingDocumentItem sample ──")
display(billing_item_df)

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS h2b_dbx_billingdocument")
spark.sql("CREATE SCHEMA IF NOT EXISTS h2b_dbx_billingdocument.billingdocument")

billing_doc_df.write.mode("overwrite").saveAsTable("h2b_dbx_billingdocument.billingdocument.billingdocument")
billing_item_df.write.mode("overwrite").saveAsTable("h2b_dbx_billingdocument.billingdocument.billingdocumentitem")

print("Written to h2b_dbx_billingdocument.billingdocument.billingdocument")
print("Written to h2b_dbx_billingdocument.billingdocument.billingdocumentitem")

In [0]:
spark.sql("ALTER TABLE h2b_dbx_billingdocument.billingdocument.billingdocument ALTER COLUMN BillingDocument SET NOT NULL")
spark.sql("ALTER TABLE h2b_dbx_billingdocument.billingdocument.billingdocument ADD CONSTRAINT billingdocument_pk PRIMARY KEY(BillingDocument)")

spark.sql("ALTER TABLE h2b_dbx_billingdocument.billingdocument.billingdocumentitem ALTER COLUMN BillingDocument SET NOT NULL")
spark.sql("ALTER TABLE h2b_dbx_billingdocument.billingdocument.billingdocumentitem ALTER COLUMN BillingDocumentItem SET NOT NULL")
spark.sql("ALTER TABLE h2b_dbx_billingdocument.billingdocument.billingdocumentitem ADD CONSTRAINT billingdocumentitem_pk PRIMARY KEY(BillingDocument, BillingDocumentItem)")